# Fine-tune NLLB-200 на ингушском языке

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**

Затем: Runtime → **Run all** (Ctrl+F9) и уходи по делам.

Модель сохранится на Google Drive в папке `My Drive/nllb-ingush/`.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "НЕТ GPU — смени runtime!")
assert torch.cuda.is_available(), "Нужен GPU!"

In [ ]:
!pip install -q transformers[torch] datasets sacrebleu sentencepiece evaluate accelerate

In [ ]:
!git clone https://github.com/Morgxth/GhalghayTools.git
%cd GhalghayTools/corpus/finetune
!echo "Train:"; wc -l < train.jsonl
!echo "Dev:";   wc -l < dev.jsonl

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
SAVE_DIR = "/content/drive/MyDrive/nllb-ingush"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Модель будет сохранена в: {SAVE_DIR}")

In [ ]:
# 88K примеров, eff batch 32 (~2750 шагов/эпоха x3 = ~3-4 часа на T4)
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

!PYTORCH_ALLOC_CONF=expandable_segments:True python train.py \
    --train   train.jsonl \
    --dev     dev.jsonl \
    --epochs  3 \
    --batch   4 \
    --grad-acc 8 \
    --lr      5e-5 \
    --max-len 96 \
    --out     "$SAVE_DIR"

In [ ]:
from transformers import pipeline

pipe_inh2rus = pipeline("translation", model=SAVE_DIR,
    src_lang="inh_Cyrl", tgt_lang="rus_Cyrl", max_length=256, device=0)
pipe_rus2inh = pipeline("translation", model=SAVE_DIR,
    src_lang="rus_Cyrl", tgt_lang="inh_Cyrl", max_length=256, device=0)

tests = [
    (pipe_inh2rus, "inh→rus", "Даьла безам бе, хьо сона везаш ва."),
    (pipe_inh2rus, "inh→rus", "Ингушетийн Республика Россе юкъе ю."),
    (pipe_inh2rus, "inh→rus", "Тахан маьрша де ду."),
    (pipe_rus2inh, "rus→inh", "Сегодня хорошая погода."),
    (pipe_rus2inh, "rus→inh", "Республика Ингушетия входит в состав России."),
]
for pipe, d, text in tests:
    res = pipe(text)[0]["translation_text"]
    print(f"[{d}] {text}")
    print(f"      {res}
")

In [ ]:
import json, evaluate
bleu = evaluate.load("sacrebleu")
rows = [json.loads(l) for l in open("test.jsonl") if l.strip()]
sample = [r for r in rows if r["src_lang"] == "inh_Cyrl"][:500]
preds = [pipe_inh2rus(r["src"])[0]["translation_text"] for r in sample]
refs  = [[r["tgt"]] for r in sample]
score = bleu.compute(predictions=preds, references=refs)
print(f"BLEU inh→rus (test-500): {score["score"]:.2f}")